#Ingesta de carpeta con archivos SCV

In [0]:
#1. Leer archivos CSV usando DataFrameReader de Spark

#Importamos las librerias que se van a utilizar
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Define el la estructura personName
movie_company_schema = StructType(fields = [
    StructField("movieId", IntegerType(), True),
    StructField("companyId", IntegerType(), True)
])

# Cargamos el archivo utilizando la estructura definida
movie_company_df = spark.read\
    .schema(movie_company_schema)\
    .csv("abfss://bronze@lsdata01.dfs.core.windows.net/movie_company")

# Mostramos el resultado
display(movie_company_df)

In [0]:
#Paso 2 - Renombrar, añadir y dar formato a las columnas requeridas
from pyspark.sql.functions import col, concat, current_timestamp, lit
movie_company_renamed_df = movie_company_df\
    .withColumnRenamed("movieId", "movie_id")\
    .withColumnRenamed("companyId", "company_id")\
    .withColumn("ingestion_date", current_timestamp())\
    .withColumn("enviroment", lit("Produccion"))
    
display(movie_company_renamed_df)


In [0]:
#Paso 4 - Guardar datos en datalake en formato parket 
movie_company_renamed_df.write.mode("overwrite").parquet("abfss://silver@lsdata01.dfs.core.windows.net/movie_companies")
df = spark.read.parquet("abfss://silver@lsdata01.dfs.core.windows.net/movie_companies")
display(df)


In [0]:
%fs
ls abfss://silver@lsdata01.dfs.core.windows.net/movie_companies